In [2]:
# [CELL 1] CYBERSECURITY STUDENT CONFIGURATION
from pathlib import Path

LAST_NAME = "Dela Cruz"
STUDENT_ID = "TUPM-1251-26"
SEED_DIGIT = 6

# Data Generation Rules from the manual:
# Target Year: 2015 + SEED_DIGIT
# Loss Field: Financial Loss (in Million $)
# Count Field / Attack Type
TARGET_YEAR = 2015 + SEED_DIGIT  # 2021

print("Target Year:", TARGET_YEAR)
print("Loss Field: Financial Loss (in Million $)")


Target Year: 2021
Loss Field: Financial Loss (in Million $)


In [3]:
# [CELL 2] INITIALIZE CYBERSECURITY WORKSPACE PATHS
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
BACKUP_DIR = Path("backup")

incident_dataset_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
filtered_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incidents_filtered.csv"
json_summary_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incidents_summary.json"
json_backup_path = BACKUP_DIR / f"{LAST_NAME}_{STUDENT_ID}_incidents_summary.json"
text_report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incidents_report.txt"

print("Dataset path:", incident_dataset_path)
print("Filtered CSV output path:", filtered_csv_path)
print("JSON summary path:", json_summary_path)
print("Text report path:", text_report_path)


Dataset path: data\Global_Cybersecurity_Threats_2015-2024.csv
Filtered CSV output path: output\Dela Cruz_TUPM-1251-26_incidents_filtered.csv
JSON summary path: output\Dela Cruz_TUPM-1251-26_incidents_summary.json
Text report path: output\Dela Cruz_TUPM-1251-26_incidents_report.txt


In [4]:
# [CELL 3] READ AND PROCESS INCIDENT DATASET
total_records = 0
target_year_records = 0
total_financial_loss = 0.0
attack_type_counts = {}
selected_incidents = []

with open(incident_dataset_path, "r", encoding="utf-8") as file:
    header = file.readline().strip().split(",")
    
    # Get column indices based on dataset schema requirements
    year_idx = header.index("Year")
    attack_idx = header.index("Attack Type")
    loss_idx = header.index("Financial Loss (in Million $)")
    
    for line in file:
        fields = line.strip().split(",")
        total_records += 1
        
        current_year = int(fields[year_idx])
        
        # 1. Read the CSV file and select incidents occurring in the target year
        if current_year == TARGET_YEAR:
            target_year_records += 1
            
            attack_type = fields[attack_idx]
            loss_val_raw = fields[loss_idx]
            
            # Handle invalid financial values safely
            try:
                financial_loss = float(loss_val_raw)
            except ValueError:
                financial_loss = 0.0
                
            total_financial_loss += financial_loss
            
            # 2. Count the incidents belonging to each attack type
            if attack_type in attack_type_counts:
                attack_type_counts[attack_type] = attack_type_counts[attack_type] + 1
            else:
                attack_type_counts[attack_type] = 1
                
            # Keep track of the original dataset order index to resolve ranking ties
            selected_incidents.append({
                "line_data": line.strip(),
                "attack_type": attack_type,
                "financial_loss": financial_loss,
                "original_index": target_year_records
            })

print("Total records in dataset:", total_records)
print("Incidents in target year:", target_year_records)


Total records in dataset: 3000
Incidents in target year: 299


In [5]:
# [CELL 4] CALCULATE METRICS AND RANK FIVE COSTLIEST INCIDENTS
# 3. Calculate the total and average financial loss of the selected incidents
if target_year_records > 0:
    average_financial_loss = total_financial_loss / target_year_records
else:
    average_financial_loss = 0.0

# 4. Identify five incidents with highest financial losses
# Use basic sorting algorithm loop matching the level of the lab manual
ranked_incidents = selected_incidents.copy()
five_costliest = []

for _ in range(5):
    if not ranked_incidents:
        break
    highest_node = None
    highest_idx = -1
    
    for idx, item in enumerate(ranked_incidents):
        if highest_node is None:
            highest_node = item
            highest_idx = idx
        else:
            # Descending financial loss rank, retain original order on ties
            if item["financial_loss"] > highest_node["financial_loss"]:
                highest_node = item
                highest_idx = idx
            elif item["financial_loss"] == highest_node["financial_loss"]:
                if item["original_index"] < highest_node["original_index"]:
                    highest_node = item
                    highest_idx = idx
                    
    if highest_node is not None:
        five_costliest.append(highest_node)
        del ranked_incidents[highest_idx]

print("Average financial loss calculated:", average_financial_loss)
print("Identified the 5 costliest incidents successfully.")


Average financial loss calculated: 53.08832775919729
Identified the 5 costliest incidents successfully.


In [6]:
# [CELL 5] EXPORT FILTERED DATA STRUCTS AND VERIFY BACKUP
import json
import shutil

# 5. Export selected incidents to a filtered CSV file
with open(filtered_csv_path, "w", encoding="utf-8") as out_csv:
    out_csv.write(",".join(header) + "\n")
    for item in selected_incidents:
        out_csv.write(item["line_data"] + "\n")

# Export top 5 list to JSON summary file
with open(json_summary_path, "w", encoding="utf-8") as out_json:
    json.dump(five_costliest, out_json, indent=4)

# Create a verified backup of the JSON file
shutil.copy2(json_summary_path, json_backup_path)

print("Filtered CSV created:", filtered_csv_path.exists())
print("Backup verification state (File exists):", json_backup_path.exists())


Filtered CSV created: True
Backup verification state (File exists): True


In [7]:
# [CELL 6] GENERATE SUMMARY TEXT REPORT
# 6. Generate a text report containing details requested by the guidelines
with open(text_report_path, "w", encoding="utf-8") as report:
    report.write("CYBERSECURITY INCIDENT EXTRACTION REPORT\n")
    report.write("========================================\n")
    report.write(f"Target Year Evaluated: {TARGET_YEAR}\n")
    report.write(f"Total Incident Count: {target_year_records}\n")
    report.write(f"Total Financial Loss: {total_financial_loss:.2f} Million USD\n")
    report.write(f"Average Financial Loss: {average_financial_loss:.2f} Million USD\n\n")
    
    report.write("Incident Distribution per Attack Type:\n")
    for attack, count in attack_type_counts.items():
        report.write(f" - {attack}: {count}\n")
        
    report.write("\nOutput File Summary Details:\n")
    report.write(f" - Filtered CSV: {filtered_csv_path.name}\n")
    report.write(f" - JSON Summary: {json_summary_path.name}\n")
    report.write(f" - Verified Backup: {json_backup_path.name}\n")

print("Text report compiled completely at:", text_report_path.name)


Text report compiled completely at: Dela Cruz_TUPM-1251-26_incidents_report.txt
